# Suite TSS — TypeScript5 simplified by ECMAScript's semantics

`TypeScript5.Simplifiers.OfAny(term, scope)` simplifies a TypeScript5 term by ECMA-262's semantics, its values and
abstract operations `TypeScript5.Abstract`'s, modeled alike in both implementations: what the scope decides folds into
the literal of its value (a negative number, NaN, Infinity, -0 and `undefined` by the names ECMAScript writes them,
`null` and a regular expression a `constant`), an array or an object of such values into its display, and the rest
stays, the residual, its subterms simplified; a short circuit decides what does not depend on what stays. What has no
rule stays too, named with its class in the evaluation inventory (`residues`): a write, a body of statements, a
coroutine (`d`), a built-in method and a host value (`c`), a type, a pattern, JSX (`b`). What ECMAScript makes an
error still raises, as JavaScript would: TypeError and RangeError, by the same messages. Terms are built through the
dialect's `make`, in a transaction (`term`); `text` gives a residual's source, as the printer writes it.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as TypeScriptPatterns from "@mbse/expressions/TypeScript/Patterns";
import * as TypeScriptRegExp from "@mbse/expressions/TypeScript/RegExp";
import * as TypeScriptTypes from "@mbse/expressions/TypeScript/Types";
import { Terms as F } from "@mbse/expressions/Framework";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
import { Builders, Errors } from "@mbse/schemas/Framework";
import { assert, equal, raises } from "./support.js";

const { NotEvaluated } = Simplification;
const { ValueError } = Errors;
const [A, TS, TV, TSText] = [TypeScript5.Abstract, TypeScript5.Expressions, TypeScript5.Simplifiers, TypeScript5.Text];

const store: any = new Builders.Store();
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install,
  TypeScript5.install, TypeScriptTypes.install, TypeScriptPatterns.install, TypeScriptRegExp.install]) {
  store.transaction(install);
}

/** A person, a reference object, with a list of scores. */
function people(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("tss.people/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject })
    .spec([(_: any) => _.Person.name("tss.Person").ref(true).properties({
      name: S.OfStr, age: S.OfInt, toString: S.OfStr, scores: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfInt])) })])]))]);
}

store.transaction(people);

/** The TypeScript5 term `spec` writes, a native value as its literal, the root of a holder built in a transaction. */
function term(spec: unknown): any {
  return store.transaction(() => F.term(store.Expressions.Framework.Expression((e: any) => e.root(TS.DIALECT.spec(spec)))));
}

/** The simplification of the term `spec` writes, in `scope`. */
const simplified = (spec: unknown, scope: Record<string, unknown> = {}) => TV.OfAny(term(spec), scope);

/** The source of the residual of the term `spec` writes, simplified in `scope`. */
function text(spec: unknown, scope: Record<string, unknown> = {}): string {
  const residual = simplified(spec, scope).residual;
  return TSText.ToText(store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root(residual))));
}

/** The value the term `spec` writes simplifies to, in `scope`: its literal root's. */
const value = (spec: unknown, scope: Record<string, unknown> = {}): any => simplified(spec, scope).evaluate();

/** Why what stayed in the residual stayed: each residue's reason and class. */
const reasons = (spec: unknown, scope: Record<string, unknown> = {}): string[][] =>
  simplified(spec, scope).residues.map((residue) => [residue.reason, residue.category]);

const make = (kind: string, attributes: Record<string, unknown>, ...args: unknown[]) => TS.DIALECT.make(new F.Form(kind, attributes, args));

// the builders of the terms the cases write
const name = (n: string) => make("identifier", { name: n });
const [x, y, c, f] = [name("x"), name("y"), name("c"), name("f")];
const binary = (op: string, a: unknown, b: unknown) => make("binaryexpression", { operator: op }, a, b);
const unary = (op: string, a: unknown) => make("unaryexpression", { operator: op }, a);
const logical = (op: string, a: unknown, b: unknown) => make("logicalexpression", { operator: op }, a, b);
const array = (...items: unknown[]) => make("arrayexpression", {}, ...items);
const hole = make("elision", {});
const spread = (a: unknown) => make("spreadelement", {}, a);
const prop = (key: unknown, v: unknown, computed = false, shorthand = false) => make("property", {
  propertyKind: "init", computed, method: false, shorthand }, key, v);
const obj = (...properties: unknown[]) => make("objectexpression", {}, ...properties);
const member = (o: unknown, p: unknown, computed = false, optional = false) => make(
  "memberexpression", { computed, optional }, o, p);
const call = (callee: unknown, ...args: unknown[]) => make("callexpression", { optional: false }, callee, ...args);
const optionalCall = (callee: unknown, ...args: unknown[]) => make("callexpression", { optional: true }, callee, ...args);
const new_ = (callee: unknown, ...args: unknown[]) => make("newexpression", {}, callee, ...args);
const chain = (e: unknown) => make("chainexpression", {}, e);
const arrow = (body: unknown, ...params: unknown[]) => make("arrowfunctionexpression", { isAsync: false }, null, body, ...params);
const asyncArrow = (body: unknown, ...params: unknown[]) => make("arrowfunctionexpression", { isAsync: true }, null, body, ...params);
const default_ = (target: unknown, v: unknown) => make("assignmentpattern", {}, target, v);
const rest = (target: unknown) => make("restelement", {}, target);
const arrayPattern = (...elements: unknown[]) => make("arraypattern", {}, ...elements);
const objectPattern = (...properties: unknown[]) => make("objectpattern", {}, ...properties);
const conditional = (t: unknown, a: unknown, b: unknown) => make("conditionalexpression", {}, t, a, b);
const sequence = (...items: unknown[]) => make("sequenceexpression", {}, ...items);
const parenthesized = (e: unknown) => make("parenthesizedexpression", {}, e);
const null_ = make("constant", { singleton: "null" }, null);
const regexp = (pattern: string, flags = "") => make("constant", { pattern, ...(flags ? { flags } : {}) }, null);
const this_ = make("thisexpression", {});
const undefined_ = name("undefined");

/** A template literal's spec: its quasis' raw text, its expressions between them. */
function template(...rawsAndExpressions: unknown[]): unknown {
  const raws = rawsAndExpressions.filter((_, i) => i % 2 === 0) as string[];
  const expressions = rawsAndExpressions.filter((_, i) => i % 2 === 1);
  return (u: any) => u.templateliteral((r: any) => r.quasis(raws.map((raw, i) => (q: any) => q.raw(raw).tail(i === raws.length - 1)))
    .expressions(expressions.map((e) => TS.DIALECT.spec(e))));
}

/** A tagged template's spec: its tag, its quasis' raw text and its expressions between them. */
function tagged(tag: unknown, ...rawsAndExpressions: unknown[]): unknown {
  const raws = rawsAndExpressions.filter((_, i) => i % 2 === 0) as string[];
  const expressions = rawsAndExpressions.filter((_, i) => i % 2 === 1);
  return (u: any) => u.taggedtemplateexpression((r: any) => r.tag(TS.DIALECT.spec(tag)).quasi((q: any) => q.quasis(
    raws.map((raw, i) => (e: any) => e.raw(raw).tail(i === raws.length - 1))).expressions(expressions.map((e) => TS.DIALECT.spec(e)))));
}

const numberType = (t: any) => t.tsnumberkeyword((k: any) => k); // a type, `TypeScript Types`' term

<a id="TSS-01"></a>
## TSS-01 · Literals, names and `this`: what the scope binds, the global object's values, and values written back as literals

In [ ]:
// A literal is its value; a name what the scope binds, else the global object's `undefined`, `NaN` and `Infinity`
assert(value(1.5) === 1.5 && value(2n) === 2n && value("a") === "a" && value(true) === true); // a number is a number, a bigint a bigint
assert(text(x, { x: 2 }) === "2" && value(x, { x: "s" }) === "s");
assert(text(undefined_) === "undefined" && simplified(undefined_).literal);
assert(text(name("NaN")) === "NaN" && text(name("Infinity")) === "Infinity");
assert(text(undefined_, { undefined: 1 }) === "1"); // a variable shadows a global
assert(value(this_, { this: 1.5 }) === 1.5 && text(this_) === "this"); // `this`, bound by the scope by that name
assert(value(null_) === null && text(null_) === "null" && String(undefined) === "undefined");
// A name the scope does not bind stays, named in `free`; its root is no literal
const unbound = simplified(binary("+", x, 1));
assert(text(binary("+", x, 1)) === "x + 1" && equal([...unbound.free], ["x"]) && unbound.residues.length === 0 && !unbound.literal);
raises(NotEvaluated, () => unbound.evaluate(), "the root is not a literal: x is not bound");
// A value is written back as ECMAScript writes it: a negative number negated, NaN, Infinity and -0 by name, `undefined`
// by name, `null` and a regular expression a constant; each a literal root
for (const [bound, written] of [[-2, "-2"], [-0, "-0"], [Infinity, "Infinity"], [-Infinity, "-Infinity"], [NaN, "NaN"],
  [-3n, "-3n"], [undefined, "undefined"], [null, "null"], ["a", '"a"'], [true, "true"],
  [new A.RegExp("a", ""), "/a/"], [new A.RegExp("a+", "gu"), "/a+/gu"]] as [unknown, string][]) {
  assert(text(x, { x: bound }) === written && simplified(x, { x: bound }).literal);
}
assert(text(unary("-", 2)) === "-2" && text(regexp("a+", "g")) === "/a+/g"); // a regular expression, as written
assert(A.typeOf(value(regexp("a"))) === "object" && value(regexp("a+", "g")).flags === "g"); // a new object

<a id="TSS-02"></a>
## TSS-02 · Conversions: ToBoolean, ToNumber of strings and objects, ToString of numbers and objects, ToPrimitive by an object's own methods

In [ ]:
// ToBoolean, by `!`
for (const [operand, negated] of [[0, true], ["", true], ["0", false], [name("NaN"), true], [0n, true], [null_, true],
  [undefined_, true], [array(), false], [obj(), false], [arrow(x, x), false], [1n, false]] as [unknown, boolean][]) {
  assert(value(unary("!", operand)) === negated);
}
// ToNumber, by `+`: StringToNumber, its white space trimmed; an object by its primitive
for (const [operand, written] of [["  12  ", "12"], ["", "0"], ["0x1F", "31"], ["0o17", "15"], ["0b101", "5"], ["-0x1", "NaN"],
  ["1_000", "NaN"], ["Infinity", "Infinity"], ["-Infinity", "-Infinity"], ["infinity", "NaN"],
  [".5", "0.5"], ["5.", "5"], ["1e3", "1000"], ["\u00a0\n7\ufeff", "7"], ["0x", "NaN"],
  ["0x" + "f".repeat(300), "Infinity"], [null_, "0"], [undefined_, "NaN"], [true, "1"],
  [array(), "0"], [array(5), "5"], [array(1, 2), "NaN"], [obj(), "NaN"]] as [unknown, string][]) {
  assert(text(unary("+", operand)) === written);
}
raises(TypeError, () => value(unary("+", 1n)), "Cannot convert a BigInt value to a number");
// ToString, by `"" + x`: Number::toString's shortest round trip, an array joined, an object's tag
for (const [operand, written] of [[1e21, "1e+21"], [1e-7, "1e-7"], [123.456, "123.456"], [binary("+", 0.1, 0.2), "0.30000000000000004"],
  [unary("-", 0), "0"], [array(1, array(2, 3)), "1,2,3"],
  [array(null_, undefined_, hole, 1), ",,,1"], [obj(), "[object Object]"],
  [regexp("a", "g"), "/a/g"], [true, "true"], [false, "false"], [10n, "10"], [null_, "null"], [undefined_, "undefined"]] as [unknown, string][]) {
  assert(value(binary("+", "", operand)) === written);
}
// ToPrimitive: an object's own valueOf, then toString (toString first for a string), where they are functions
assert(value(binary("+", obj(prop(name("valueOf"), arrow(5))), 1)) === 6);
assert(value(template("", obj(prop(name("toString"), arrow("s")), prop(name("valueOf"), arrow(1))), "")) === "s");
assert(value(binary("+", obj(prop(name("valueOf"), arrow(obj())), prop(name("toString"), arrow("t"))), "")) === "t");
raises(TypeError, () => value(binary("+", obj(prop(name("toString"), 1)), "")), "Cannot convert object to primitive value");
// A function's source text is not modeled: it stays, named
assert(equal(reasons(binary("+", arrow(x, x), "")), [["TypeScript5 Function.prototype.toString is not simplified", "c"]]));

<a id="TSS-03"></a>
## TSS-03 · Numbers: IEEE 754 arithmetic, division by zero, remainder and exponentiation as ECMAScript defines them, and the 32-bit operators

In [ ]:
const minusZero = unary("-", 0);
for (const [left, op, right, written] of [
  [1, "+", 2, "3"], [0.1, "+", 0.2, "0.30000000000000004"], [7, "-", 10, "-3"], [2, "*", 3, "6"],
  [1, "/", 0, "Infinity"], [-1, "/", 0, "-Infinity"], [1, "/", minusZero, "-Infinity"], [0, "/", 0, "NaN"],
  [name("NaN"), "/", 0, "NaN"], [5, "%", 3, "2"], [-5, "%", 3, "-2"], [5, "%", -3, "2"],
  [-4, "%", 2, "-0"], [5, "%", 0, "NaN"], [name("Infinity"), "%", 2, "NaN"], [5, "%", name("Infinity"), "5"],
  [0, "%", 5, "0"], [5.5, "%", 2, "1.5"], [2, "**", 10, "1024"], [2, "**", -1, "0.5"],
  [-8, "**", binary("/", 1, 3), "NaN"], [1, "**", name("Infinity"), "NaN"], [name("NaN"), "**", 0, "1"],
  [2, "**", name("NaN"), "NaN"], [name("NaN"), "**", 2, "NaN"], [0, "**", -1, "Infinity"],
  [minusZero, "**", -1, "-Infinity"], [minusZero, "**", 3, "-0"], [minusZero, "**", 2, "0"],
  [0, "**", 2, "0"], [10, "**", 400, "Infinity"], [-10, "**", 401, "-Infinity"], [4, "**", 0.5, "2"],
  [-2, "**", 3, "-8"], [5, "&", 3, "1"], [5, "|", 3, "7"], [5, "^", 3, "6"],
  [1, "<<", 31, "-2147483648"], [1, "<<", 32, "1"], [-16, ">>", 2, "-4"], [-16, ">>>", 28, "15"],
  [-1, ">>>", 0, "4294967295"], [4294967301, "|", 0, "5"], [name("NaN"), "|", 0, "0"],
  [3.7, "|", 0, "3"], [-3.7, "|", 0, "-3"], [name("Infinity"), "|", 0, "0"],
  ["3", "*", "4", "12"], ["3", "-", 1, "2"], ["a", "*", 1, "NaN"], [true, "+", 1, "2"], [null_, "+", 1, "1"],
  [undefined_, "+", 1, "NaN"], [array(2), "*", array(3), "6"]] as [unknown, string, unknown, string][]) {
  assert(text(binary(op, left, right)) === written, `${op} ${written}`);
}
for (const [op, operand, written] of [["~", 5, "-6"], ["-", "3", "-3"], ["-", null_, "-0"], ["-", minusZero, "0"],
  ["~", "x", "-1"], ["-", array(), "-0"]] as [string, unknown, string][]) {
  assert(text(unary(op, operand)) === written);
}

<a id="TSS-04"></a>
## TSS-04 · BigInt: arbitrary precision, truncating division, its errors, and its comparisons with numbers and strings

In [ ]:
for (const [left, op, right, written] of [
  [1n, "+", 2n, "3n"], [2n, "**", 64n, "18446744073709551616n"], [7n, "/", 2n, "3n"], [-7n, "/", 2n, "-3n"], [7n, "%", -2n, "1n"],
  [-7n, "%", 2n, "-1n"], [5n, "-", 8n, "-3n"], [3n, "*", -4n, "-12n"], [5n, "&", 3n, "1n"], [-5n, "|", 2n, "-5n"],
  [5n, "^", -1n, "-6n"], [1n, "<<", 100n, "1267650600228229401496703205376n"], [256n, ">>", 4n, "16n"], [-9n, ">>", 1n, "-5n"],
  [1n, "<<", -1n, "0n"], [8n, ">>", -1n, "16n"], [0n, "**", 0n, "1n"], [1n, "**", 2n ** 40n, "1n"], [0n, "<<", 2n ** 40n, "0n"],
] as [unknown, string, unknown, string][]) {
  assert(text(binary(op, left, right)) === written, `${op} ${written}`);
}
assert(text(unary("-", 5n)) === "-5n" && text(unary("~", 5n)) === "-6n" && text(unary("-", unary("-", 5n))) === "5n");
for (const [spec, error, message] of [
  [binary("+", 1n, 1), TypeError, "Cannot mix BigInt and other types, use explicit conversions"],
  [binary("/", 1n, 0n), RangeError, "Division by zero"], [binary("%", 1n, 0n), RangeError, "Division by zero"],
  [binary("**", 2n, -1n), RangeError, "Exponent must be non-negative"],
  [binary(">>>", 1n, 0n), TypeError, "BigInts have no unsigned right shift, use >> instead"],
  [binary("**", 2n, 2n ** 40n), RangeError, "Maximum BigInt size exceeded"],
  [binary("<<", 1n, 2n ** 31n), RangeError, "Maximum BigInt size exceeded"]] as [unknown, Function, string][]) {
  raises(error, () => value(spec), message);
}
// Compared with numbers mathematically, with strings by StringToBigInt (undefined where a string is no integer)
for (const [left, op, right, result] of [
  [1n, "<", 2, true], [2n, ">", "1", true], [1n, "==", "1", true], [1n, "==", "x", false], [1n, "<", "x", false],
  ["x", ">", 1n, false], ["1", "<", 2n, true], ["x", "<", 1n, false], [1n, "==", 1, true], [1n, "===", 1, false],
  [1n, "==", 1.5, false], [1n, "==", name("Infinity"), false], [name("NaN"), "<", 1n, false], [2n, ">", 1.5, true],
  ["0x10", "==", 16n, true], [" 12 ", "==", 12n, true], ["", "==", 0n, true], ["1.0", "==", 1n, false],
  ["-0x10", "==", -16n, false], ["-5", "==", -5n, true], [2n ** 64n, ">", 1e19, true], [1, "==", 1n, true],
  [1n, "<", "2", true]] as [unknown, string, unknown, boolean][]) {
  assert(value(binary(op, left, right)) === result, `${String(left)} ${op} ${String(right)}`);
}

<a id="TSS-05"></a>
## TSS-05 · Equality and order: IsLooselyEqual, IsStrictlyEqual and IsLessThan, objects by identity, strings by UTF-16 code units

In [ ]:
const same = arrow(binary("===", x, x), x); // an object is itself
for (const [left, op, right, result] of [
  [null_, "==", undefined_, true], [null_, "===", undefined_, false], [null_, "===", null_, true], [null_, "==", 0, false], [undefined_, "==", 0, false],
  [name("NaN"), "==", name("NaN"), false], [name("NaN"), "===", name("NaN"), false], [0, "===", minusZero, true],
  ["1", "==", 1, true], ["", "==", 0, true], ["a", "==", "a", true], [true, "==", 1, true], [true, "==", "1", true],
  [1, "==", false, false], [array(1), "==", 1, true], [array(1), "==", "1", true], [array(), "==", "", true],
  ["1", "==", array(1), true], [obj(), "==", "[object Object]", true], [array(), "==", array(), false],
  [obj(), "===", obj(), false], [null_, "==", false, false], [1, "!=", "1", false], [1, "!==", "1", true],
  ["b", ">", "a", true], ["B", "<", "a", true], ["a", "<", "aa", true], ["\u{10000}", "<", "\u{ffff}", true],
  ["10", "<", "9", true], ["10", "<", 9, false], [1, "<", name("NaN"), false], [name("NaN"), ">=", 1, false],
  [1, "<=", 1, true], [2, ">=", 3, false], [null_, ">=", 0, true], [undefined_, ">=", 0, false],
  [array(2), ">", 1, true], [1, "<=", name("NaN"), false], [2, ">", 1, true]] as [unknown, string, unknown, boolean][]) {
  assert(value(binary(op, left, right)) === result, `${op} ${result}`);
}
assert(value(call(same, array())) === true && value(call(same, obj())) === true);

<a id="TSS-06"></a>
## TSS-06 · Strings and templates: UTF-16 code units, cooked and raw strings, and a tag called with them

In [ ]:
const [high, low] = [String.fromCharCode(0xd83d), String.fromCharCode(0xde00)];
assert(value(binary("+", "a", "b")) === "ab" && value(member("\u{1F600}", name("length"))) === 2);
assert(value(member("\u{1F600}", 0, true)) === high); // a code unit, a lone surrogate
assert(value(binary("+", high, low)) === "\u{1F600}" && value(binary("===", binary("+", high, low), "\u{1F600}")));
assert(text(array(spread("a\u{1F600}"))) === '["a", "\u{1F600}"]'); // a string's iterator gives code points
assert(value(template("a", binary("+", 1, 1), "b")) === "a2b" && value(template("", array(1, 2), "")) === "1,2");
assert(value(template("", null_, "")) === "null" && text(template("a", x, "")) === "`a${x}`");
assert(value((u: any) => u.templateliteral((r: any) => r.quasis([(q: any) => q.tail(true)]))) === ""); // a quasi with no raw text
// A quasi's escapes cooked: single, hexadecimal, Unicode, NUL, any other character itself, a line continuation nothing;
// a line terminator read as LF
assert(value(template(String.raw`\n\t\x41\u0042\u{43}\0\'\`\$\\\q`)) === "\n\tABC\0'`$\\q");
assert(value(template("a\\\nb\r\nc\rd\\\u{2028}e")) === "ab\nc\nde");
for (const raw of ["\\1", "\\08", "\\x4", "\\u{110000}", "\\u12", "\\"]) {
  raises(ValueError, () => value(template(raw)), "a templateliteral's quasi has an invalid escape");
}
// A tag is called with its strings, an array whose `raw` is their raw text, undefined where an escape is invalid, then
// the values between them
const [s, v] = [name("s"), name("v")];
const tag = arrow(array(member(member(s, name("raw")), 1, true), member(s, 1, true),
  member(v, 0, true), binary("in", "raw", s), member(s, name("length"))), s, rest(v));
assert(text(tagged(tag, "a", 1, "b\\xZ")) === '["b\\\\xZ", undefined, 1, true, 2]');
assert(text(tagged(tag, "a", x, "b")) === "((s, ...v) => [s.raw[1], s[1], v[0], \"raw\" in s, s.length])`a${x}b`");
assert(text(tagged(arrow(s, s), "a")) === "((s) => s)`a`"); // its strings have no display
raises(TypeError, () => value(tagged(1, "a")), "1 is not a function");

<a id="TSS-07"></a>
## TSS-07 · Arrays and objects: holes, spreads and displays, keys in ECMAScript's order, member access, optional chains, `in`, data objects and host values

In [ ]:
// Arrays: holes, spreads by iteration; a display of literals, or the subterms simplified
assert(text(array(1, hole, 2)) === "[1,, 2]" && text(array(spread(array(1, hole, 2)))) === "[1, undefined, 2]");
assert(text(array(1, spread(x), binary("+", 2, 3))) === "[1, ...x, 5]" && text(array(hole, x)) === "[, x]");
assert(text(array(spread(array(1, 2)), x)) === "[...[1, 2], x]" && text(array(arrow(x, x))) === "[(x) => x]");
// Objects: keys in ECMAScript's order, array indices first; computed keys; shorthands; spreads; a later key's value
assert(text(obj(prop(name("b"), 1), prop(name("a"), 2), prop(1, 3), prop(0, 4))) === "({ 0: 4, 1: 3, b: 1, a: 2 })");
assert(text(obj(prop(binary("+", "a", "b"), 1, true), prop(x, x, false, true)), { x: 1 }) === "({ ab: 1, x: 1 })");
assert(text(obj(prop(1.5, 1), prop("a-b", null_), prop(2n, true))) === '({ 2: true, "1.5": 1, "a-b": null })');
assert(text(obj(prop(name("a"), 1), prop(name("b"), 2), prop(name("a"), 3))) === "({ a: 3, b: 2 })");
assert(text(obj(spread(obj(prop(name("a"), 1))), prop(name("b"), 2), spread(array(7)), spread("hi"), spread(null_),
  spread(5))) === '({ 0: "h", 1: "i", a: 1, b: 2 })');
assert(text(obj(prop(name("a"), binary("+", 1, 1)), prop(name("b"), x))) === "({ a: 2, b: x })");
assert(text(obj(prop(name("a"), arrow(x, x)))) === "({ a: (x) => x })"); // a closure has no literal
assert(text(obj(prop("__proto__", 1, true))) === '({ ["__proto__"]: 1 })'); // a property, no prototype
assert(equal(reasons(obj(prop(name("__proto__"), null_))), [["TypeScript5 objectexpression __proto__ is not simplified", "a"]]));
// Member access: own properties, else undefined; a built-in method read but not modeled
for (const [spec, written] of [
  [member(array(1, 2), name("length")), "2"], [member(array(1, 2), 1, true), "2"],
  [member(array(1, 2), 5, true), "undefined"], [member(array(1, hole), 1, true), "undefined"],
  [member(array(1, 2), "1", true), "2"], [member(array(1), name("foo")), "undefined"],
  [member("abc", name("length")), "3"], [member("abc", 1, true), '"b"'], [member("abc", 5, true), "undefined"],
  [member(obj(prop(name("a"), 1)), name("a")), "1"], [member(obj(prop(name("a"), 1)), name("b")), "undefined"],
  [member(obj(prop(name("toString"), 1)), name("toString")), "1"], [member(1, name("foo")), "undefined"],
  [member(arrow(x, x), name("foo")), "undefined"], [member(regexp("a"), name("foo")), "undefined"]] as [unknown, string][]) {
  assert(text(spec) === written, written);
}
for (const [spec, builtin] of [[member("abc", name("toUpperCase")), "String.prototype.toUpperCase"],
  [member(array(), name("map")), "Array.prototype.map"],
  [member(obj(), name("toString")), "Object.prototype.toString"],
  [member(1, name("toFixed")), "Number.prototype.toFixed"],
  [member(true, name("valueOf")), "Boolean.prototype.valueOf"],
  [member(1n, name("toString")), "BigInt.prototype.toString"],
  [member(arrow(x, x), name("call")), "Function.prototype.call"],
  [member(regexp("a"), name("test")), "RegExp.prototype.test"],
  [member(array(), name("hasOwnProperty")), "Object.prototype.hasOwnProperty"]] as [unknown, string][]) {
  assert(equal(reasons(spec), [[`TypeScript5 ${builtin} is not simplified`, "c"]]));
}
assert(text(call(member("abc", name("toUpperCase")))) === '"abc".toUpperCase()');
for (const [spec, message] of [[member(null_, name("a")), "Cannot read properties of null (reading 'a')"],
  [member(undefined_, 0, true), "Cannot read properties of undefined (reading '0')"],
  [chain(member(member(obj(prop(name("a"), null_)), name("a"), false, true), name("b"))),
    "Cannot read properties of null (reading 'b')"],
  [binary("in", 1, "abc"), "Cannot use 'in' operator to search for '1' in abc"]] as [unknown, string][]) {
  raises(TypeError, () => value(spec), message);
}
// An optional chain short-circuits to undefined where `?.` meets null or undefined
assert(text(chain(member(null_, name("a"), false, true))) === "undefined");
assert(text(chain(member(member(null_, name("a"), false, true), name("b")))) === "undefined");
assert(text(chain(optionalCall(null_, 1))) === "undefined" && text(chain(member(x, name("a"), false, true))) === "x?.a");
assert(text(chain(optionalCall(arrow(1)))) === "1");
// `in`: own and inherited properties
for (const [key, operand, result] of [["length", array(), true], [0, array(1), true], [1, array(1, hole), false],
  [2, array(1), false], ["map", array(), true], ["x", array(), false], ["raw", array(), false],
  ["hasOwnProperty", obj(), true], ["a", obj(prop(name("a"), 1)), true], ["b", obj(), false],
  ["length", arrow(x, x), true], ["foo", arrow(x, x), false], ["lastIndex", regexp("a"), true],
  ["x", regexp("a"), false]] as [unknown, unknown, boolean][]) {
  assert(value(binary("in", key, operand)) === result, `${String(key)} ${result}`);
}
// A data object's properties, as an object's: a list an array
const [ann, nobody] = store.transaction(() => [
  store.tss.Person((p: any) => p.name("Ann").age(30n).scores([1n, 2n])),
  store.tss.Person((p: any) => p),
]);
const p = name("p");
for (const [spec, written] of [[member(p, name("name")), '"Ann"'], [member(p, name("age")), "30n"], [member(p, name("scores")), "[1n, 2n]"],
  [member(p, name("missing")), "undefined"], [binary("in", "name", p), "true"], [unary("typeof", p), '"object"'],
  [obj(spread(p)), '({ name: "Ann", age: 30n, scores: [1n, 2n] })'], [binary("+", p, ""), '"[object Object]"'],
  [binary("===", p, p), "true"], [binary("==", p, "[object Object]"), "true"]] as [unknown, string][]) {
  assert(text(spec, { p: ann }) === written, written);
}
assert(value(binary("in", "age", p), { p: nobody }) === false && text(p, { p: ann }) === "p"); // an object has no literal
assert(equal(reasons(member(p, name("valueOf")), { p: ann }), [["TypeScript5 Object.prototype.valueOf is not simplified", "c"]]));
assert(value(binary("+", obj(prop(name("valueOf"), arrow(1))), p), { p: ann }) === "1[object Object]");
const named = store.transaction(() => store.tss.Person((p: any) => p.toString("me"))); // its own `toString`, a string, no
raises(TypeError, () => value(binary("+", p, ""), { p: named }), "Cannot convert object to primitive value"); // function: no primitive
// A host value, any other the scope provides, has nothing ECMAScript reads: it stays, named
for (const spec of [member(x, name("a")), binary("+", x, ""), binary("in", "a", x), array(spread(x)), obj(spread(x))]) {
  assert(equal([...new Set(reasons(spec, { x: new Map() }).map((r) => r.join("|")))], ["TypeScript5 host value is not simplified|c"]));
}
assert(value(unary("typeof", x), { x: new Map() }) === "object");

<a id="TSS-08"></a>
## TSS-08 · Arrows, calls and `new`: closures binding their parameters by destructuring, the scope's functions and classes, and what stays

In [ ]:
const [a, b, r] = [name("a"), name("b"), name("r")];
for (const [spec, written] of [
  [call(arrow(binary("+", x, 1), x), 2), "3"], [call(arrow(binary("+", x, y), x, default_(y, 10)), 1), "11"],
  [call(arrow(y, x, default_(y, x)), 5), "5"], [call(arrow(y, x, default_(y, 10)), 1, 2), "2"], [call(arrow(r, rest(r)), 1, 2), "[1, 2]"],
  [call(arrow(member(r, name("length")), a, rest(r)), 1, 2, 3), "2"], [call(arrow(b, a, b), 1), "undefined"],
  [call(arrow(array(a, b, c), arrayPattern(a, hole, default_(b, 9), rest(c))), array(1, 2, undefined_, 4, 5)),
    "[1, 9, [4, 5]]"],
  [call(arrow(array(a, b), arrayPattern(a, b)), array(1)), "[1, undefined]"],
  [call(arrow(array(a, c, r), objectPattern(prop(a, a, false, true), prop(b, objectPattern(prop(c, c, false, true))),
    rest(r))), obj(prop(a, 1), prop(b, obj(prop(c, 2))), prop(name("d"), 3))),
  "[1, 2, { d: 3 }]"],
  [call(arrow(a, objectPattern(prop(binary("+", "a", ""), a, true))), obj(prop(a, 7))), "7"],
  [call(arrow(a, objectPattern(prop("a", a))), obj(prop(a, 8))), "8"],
  [call(arrow(binary("+", a, b), a, b), spread(array(1, 2))), "3"], [call(arrow(r, rest(r)), spread("ab")), '["a", "b"]'],
  [call(call(arrow(arrow(binary("+", x, y), y), x), 1), 2), "3"], [call(arrow(this_)), "(() => this)()"],
] as [unknown, string][]) {
  assert(text(spec) === written, written);
}
assert(value(call(arrow(this_)), { this: 4 }) === 4); // `this`, an arrow's, its scope's
// A closure has no literal: it stays, its body simplified, its parameters unknown within it
assert(text(arrow(x, x)) === "(x) => x" && text(arrow(binary("+", y, binary("+", 1, 2)), y)) === "(y) => y + 3");
assert(text(arrow(binary("+", x, name("z")), x), { z: 1 }) === "(x) => x + 1");
assert(text(call(arrow(binary("+", x, y), x), 2)) === "((x) => x + y)(2)");
for (const [spec, message] of [[call(arrow(a, objectPattern(prop(a, a, false, true))), null_), "Cannot destructure 'null' as it is null."],
  [call(arrow(a, arrayPattern(a)), 5), "number is not iterable"],
  [call(arrow(a, arrayPattern(a)), null_), "null is not iterable"],
  [array(spread(obj())), "object is not iterable"], [call(1, 2), "1 is not a function"],
  [call(member(x, name("y"))), "x.y is not a function"], [new_(arrow(x, x)), "(x) => x is not a constructor"],
  [new_(1), "1 is not a constructor"]] as [unknown, string][]) {
  raises(TypeError, () => value(spec, { x: { y: 1 } }), message);
}
raises(ValueError, () => value(call(arrow(1, member(x, name("y"))), 1)), "a memberexpression is not a parameter");
// A function and a class the scope provides are called, and an object made by one is an instance of it
class Point {
  constructor(readonly at: unknown) {}
}

const twice = (n: number) => n * 2;
const scope = { f: twice, Point };
assert(value(call(f, 21), scope) === 42 && value(unary("typeof", f), scope) === "function" && text(f, scope) === "f");
assert(value(binary("instanceof", new_(name("Point"), 1), name("Point")), scope) === true);
assert(value(binary("instanceof", obj(), name("Point")), scope) === false && value(binary("instanceof", 1, f), scope) === false);
assert(value(unary("typeof", new_(name("Point"), 1)), scope) === "object");
for (const [spec, message] of [[new_(f), "f is not a constructor"],
  [binary("instanceof", obj(), f), "Function has non-object prototype 'undefined' in instanceof check"],
  [binary("instanceof", array(), arrow(x, x)), "Function has non-object prototype 'undefined' in instanceof check"],
  [binary("instanceof", obj(), 1), "Right-hand side of 'instanceof' is not callable"]] as [unknown, string][]) {
  raises(TypeError, () => value(spec, scope), message);
}
// An async arrow's call and a block body's stay, named: a promise and statements are mbse-programs'
assert(value(unary("typeof", asyncArrow(x, x))) === "function");
assert(equal(reasons(call(asyncArrow(x, x), 1)), [["TypeScript5 async arrowfunctionexpression is not simplified", "d"]]));
assert(equal(reasons(call(arrow(null), 1)), [["TypeScript5 arrowfunctionexpression block is not simplified", "d"]]));

<a id="TSS-09"></a>
## TSS-09 · Logical, conditional and sequence operators, short circuits around what stays, TypeScript's assertions erased, and what stays, by its class

In [ ]:
for (const [spec, written] of [
  [logical("&&", 1, 2), "2"], [logical("&&", 0, x), "0"], [logical("||", "", "d"), '"d"'], [logical("??", null_, 5), "5"],
  [logical("??", 0, 5), "0"], [logical("&&", true, x), "x"], [logical("||", false, x), "x"], [logical("??", undefined_, x), "x"],
  [logical("&&", x, 1), "x && 1"], [logical("||", 1, x), "1"], [logical("??", 1, x), "1"], [logical("||", x, true), "x || true"],
  [logical("||", arrow(x, x), y), "((x) => x) || y"],
  [conditional(1, "a", "b"), '"a"'], [conditional(c, 1, 2), "c ? 1 : 2"], [conditional(true, x, y), "x"],
  [conditional(0, x, y), "y"], [sequence(1, 2), "2"], [sequence(1, x), "x"], [sequence(x, 1), "x, 1"],
  [parenthesized(binary("+", 1, 2)), "3"], [parenthesized(x), "(x)"], [unary("void", 1), "undefined"],
  [unary("typeof", null_), '"object"'], [unary("typeof", 1n), '"bigint"'], [unary("typeof", undefined_), '"undefined"'],
  [unary("typeof", x), "typeof x"]] as [unknown, string][]) {
  assert(text(spec) === written, written);
}
// TypeScript's assertions are erased: their operand's value
for (const [kind, args] of [["tsasexpression", [x, numberType]], ["tssatisfiesexpression", [x, numberType]],
  ["tstypeassertion", [numberType, x]], ["tsnonnullexpression", [x]],
  ["tsinstantiationexpression", null]] as [string, unknown[] | null][]) {
  const spec = args !== null ? make(kind, {}, ...args) : (u: any) => u.tsinstantiationexpression( // its helper's type
    (r: any) => r.expression(x).typeArguments((a: any) => a.params([numberType])));
  assert(value(spec, { x: 1.5 }) === 1.5);
}
// A type a residual keeps is `TypeScript Types`', which simplification across a sub-dialect is to reach (CAP-299)
assert(text(make("tsasexpression", {}, x, numberType)) === "x as number");
assert(equal(reasons(make("tsasexpression", {}, x, numberType)), [["TypeScript Types tsnumberkeyword is not simplified", "b"]]));
// What stays, by its class: a write, an update, `delete`, `#x` (d); a signature without a body (b)
for (const [spec, reason] of [
  [make("assignmentexpression", { operator: "=" }, (p: any) => p.identifier((i: any) => i.name("x")), 1), ["TypeScript5 assignmentexpression is not simplified", "d"]],
  [make("updateexpression", { operator: "++", prefix: true }, x), ["TypeScript5 updateexpression is not simplified", "d"]],
  [unary("delete", member(x, name("a"))), ["TypeScript5 unaryexpression 'delete' is not simplified", "d"]],
  [member(this_, make("privateidentifier", { name: "x" })), ["TypeScript5 privateidentifier is not simplified", "d"]],
  [make("tsemptybodyfunctionexpression", { isAsync: false, generator: false }, null, null), ["TypeScript5 tsemptybodyfunctionexpression is not simplified", "b"]],
] as [unknown, string[]][]) {
  assert(equal(reasons(spec, { this: {} })[0], reason), reason[0]);
}
assert(equal([...TV.FOLDED].sort(), ["literal", "constant", "identifier", "thisexpression", "templateliteral", "taggedtemplateexpression",
  "parenthesizedexpression", "sequenceexpression", "elision", "spreadelement", "arrayexpression",
  "property", "objectexpression", "memberexpression", "chainexpression", "callexpression",
  "newexpression", "arrowfunctionexpression", "unaryexpression", "binaryexpression",
  "logicalexpression", "conditionalexpression", "tsasexpression", "tssatisfiesexpression",
  "tstypeassertion", "tsnonnullexpression", "tsinstantiationexpression"].sort()));
assert(TV.SIMPLIFIER.category("TypeScript5", "String.prototype.at", "memberexpression") === "c" && A.BUILTINS.length === 139);